# BERTurk fine-tune + ONNX INT8 quantization (Colab, GPU)

Bu notebook sadece **Transformer kısmını** eğitir; diğer modeller (lexicon, tf-idf, word2vec+FFN) kendi bilgisayarında CPU'da eğitilir.

Akış: kodu çek → veri → hızlı smoke test → tam eğitim (3 epoch) → ONNX FP32 export → INT8 quantization → Hugging Face Hub'a yükle.

**Runtime → Change runtime type → A100 GPU** seç.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Kodu al
GitHub reponun linkini yaz. (Repo private ise: Colab'a zip olarak yükleyip `!unzip` de yapabilirsin.)

In [ ]:
REPO_URL = "https://github.com/<kullanici-adin>/chat-moderation.git"  # <-- değiştir
!git clone -q $REPO_URL /content/chat-moderation
%cd /content/chat-moderation
!pip install -q onnx onnxruntime onnxscript

## 2. Veri
OffensEval-TR 2020 indirilir; train resmi eğitim dosyasının %90'ı, val %10'u (tweet id hash'ine göre, her makinede aynı), test resmi test seti.

In [ ]:
!python -m src.data

## 3. Smoke test (1-2 dakika)
1000 örnekle 1 epoch: kodun uçtan uca çalıştığını görmek için. Burada hata çıkarsa tam eğitime geçme, hatayı paylaş.

In [ ]:
!python -m src.train_bert --quick

## 4. Tam eğitim
BERTurk + sınıflandırma başlığı, AdamW lr=2e-5, batch 32, 3 epoch, bf16. Her epoch sonunda val macro-F1 ölçülür, en iyi checkpoint ve karar eşiği (threshold) kaydedilir.

In [ ]:
!python -m src.train_bert --epochs 3 --lr 2e-5 --batch 32

## 5. ONNX export + INT8 quantization
- FP32 ONNX export, ardından PyTorch ile aynı çıktıyı verdiği kontrol edilir (assert)
- ONNX Runtime **dynamic quantization**: ağırlıklar INT8, aktivasyonlar çalışma anında quantize edilir
- INT8 dosyasının gerçekten küçüldüğü kontrol edilir, iki modelin de test macro-F1'i yazdırılır

Latency burada ölçülmez: nihai tablo kendi bilgisayarında `python -m src.benchmark` ile çıkar.

In [ ]:
!python -m src.export_onnx
!ls -lh models/berturk_onnx

## 6. Hugging Face Hub'a yükle
Önce huggingface.co'da hesap aç ve **write** yetkili bir token oluştur. Aşağıdaki hücrede token'ı kendin gir. Sonra `HF_REPO` adını yaz.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
HF_REPO = "<hf-kullanici-adin>/berturk-offenseval-tr-onnx"  # <-- değiştir
!python scripts/upload_models.py --repo $HF_REPO

### (Alternatif) Google Drive'a kopyala
HF kullanmak istemezsen klasörü Drive'a kopyala, bilgisayarında `models/berturk_onnx/` içine koy.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/chat-moderation && cp -r models/berturk_onnx /content/drive/MyDrive/chat-moderation/